In [ ]:
import pandas as pd
from shapely.geometry import Point
from geopandas import GeoDataFrame
import requests
import sqlite3

In [ ]:
server = "https://geoservices.wallonie.be/arcgis/rest/services/EAU/BAIGNADE/MapServer/0/query?f=json&outfields=*&where=1%3D1"

In [ ]:
data = requests.get(server)
crs = data.json()['spatialReference']['wkid']
locations = data.json()['features']

In [ ]:
crs

In [ ]:
locations

In [ ]:
df = pd.json_normalize(locations)
df = df[[
    'attributes.BWID', 'attributes.NOM', 'attributes.COMMUNE', 'geometry.x', 'geometry.y'
]]
df

In [ ]:
geometry = [Point(x, y)
            for x, y in zip(df['geometry.x'], df['geometry.y'])]
for item in geometry:
    print(item.x, item.y)


In [ ]:
gdf = GeoDataFrame(df, geometry=geometry).set_crs(crs)
gdf = gdf.to_crs(4326)
gdf['lat'], gdf['lon'] = [item.y for item in gdf['geometry']], [
    item.x for item in gdf['geometry']]
gdf = gdf.drop(columns=[
    "geometry.x", "geometry.y", "geometry"
]).rename(columns={
    "attributes.BWID": "id",
    "attributes.NOM": "name",
    "attributes.COMMUNE": "alternate_name"
}).set_index("id")
gdf

In [ ]:
db = sqlite3.connect("../dataset.sqlite3")
gdf.to_sql("locations", db, if_exists="append")
